
# Lasso Penalty Simulation

This notebook is intentionally self-contained. It does not import any local utility module, so you can open this file alone during study and see the full statistical workflow, simulation setup, model fitting, evaluation, plotting, and interpretation pattern in one place.



## What problem are we solving?

Lasso regression predicts a numeric response while shrinking coefficients, often setting irrelevant predictors exactly to zero.

**Highest-probability exam pattern:** Simulate many predictors with only a few true signals, use CV to choose alpha, and interpret shrinkage and variable selection.



## Assumptions, intuition, and theory

The lasso minimizes squared error plus an L1 penalty. Standardization matters because the penalty is applied to coefficient size. Larger alpha means stronger shrinkage and simpler models.



    ## Python method notes and documentation

    `StandardScaler` puts predictors on comparable scales, `LassoCV` chooses alpha by cross-validation, `coef_` stores fitted coefficients, and `predict` gives held-out numeric predictions.

    Documentation links:

    - [NumPy random generator](https://numpy.org/doc/stable/reference/random/generator.html)
- [pandas DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)
- [matplotlib pyplot](https://matplotlib.org/stable/api/pyplot_summary.html)
- [Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html)
- [LassoCV](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LassoCV.html)
- [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html)
- [make_pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.make_pipeline.html)
- [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)
- [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)
- [mean_squared_error](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html)
- [accuracy_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html)


## Fully self-contained worked example

Every code line below is commented so the workflow can be scanned quickly under exam time pressure.

In [ ]:
import numpy as np  # Import numerical arrays and random-number tools.
import pandas as pd  # Import DataFrame tables for alpha and coefficient summaries.
import matplotlib.pyplot as plt  # Import plotting tools for coefficient shrinkage.
from sklearn.linear_model import Lasso, LassoCV  # Import lasso estimators.
from sklearn.metrics import mean_squared_error  # Import regression prediction error metric.
from sklearn.model_selection import train_test_split  # Import train/test splitting.
from sklearn.pipeline import make_pipeline  # Import pipeline construction.
from sklearn.preprocessing import StandardScaler  # Import standardization for penalty fairness.
RANDOM_SEED = 123  # Store the reproducibility seed.
rng = np.random.default_rng(RANDOM_SEED)  # Create a reproducible random generator.
plt.style.use('default')  # Use a predictable plotting style.


In [ ]:
def make_sparse_regression_data(n=140, p=18, noise=1.0, random_state=123):  # Define a sparse high-dimensional regression simulator.
    rng = np.random.default_rng(random_state)  # Create a reproducible random generator.
    X = rng.normal(0.0, 1.0, size=(n, p))  # Generate standardized-looking predictors.
    beta = np.zeros(p)  # Start with all coefficients equal to zero.
    beta[:4] = np.array([3.0, -2.0, 1.5, -1.0])  # Make only the first four predictors truly active.
    y = X @ beta + rng.normal(0.0, noise, size=n)  # Generate the response from sparse signal plus noise.
    return X, y, beta  # Return predictors, response, and true coefficients.


In [ ]:
X, y, true_beta = make_sparse_regression_data(n=150, p=18, noise=1.2, random_state=RANDOM_SEED)  # Simulate a sparse regression problem.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=RANDOM_SEED)  # Split data into training and testing sets.
lasso_cv = make_pipeline(StandardScaler(), LassoCV(alphas=np.logspace(-3, 1, 60), cv=5, max_iter=20000, random_state=RANDOM_SEED))  # Build standardized lasso with CV alpha selection.
lasso_cv.fit(X_train, y_train)  # Fit lasso using training data only.
test_pred = lasso_cv.predict(X_test)  # Predict held-out responses.
test_mse = mean_squared_error(y_test, test_pred)  # Compute held-out MSE.
selected_alpha = lasso_cv.named_steps['lassocv'].alpha_  # Extract the CV-selected alpha.
selected_coefs = lasso_cv.named_steps['lassocv'].coef_  # Extract fitted coefficients on the standardized scale.
coef_table = pd.DataFrame({'feature': np.arange(1, len(selected_coefs) + 1), 'true_beta': true_beta, 'lasso_coef': selected_coefs})  # Build a coefficient comparison table.
display(coef_table)  # Display true and estimated coefficients.
print(f'Selected alpha: {selected_alpha:.4f}')  # Print the selected penalty strength.
print(f'Test MSE: {test_mse:.3f}')  # Print held-out prediction error.
plt.figure(figsize=(7, 4))  # Create a coefficient plot.
plt.axhline(0.0, color='black', linewidth=1)  # Add a zero line for shrinkage reference.
plt.bar(coef_table['feature'], coef_table['lasso_coef'], alpha=0.75, label='lasso coefficient')  # Plot estimated lasso coefficients.
plt.scatter(coef_table['feature'], coef_table['true_beta'], color='red', label='true coefficient')  # Overlay true coefficients.
plt.xlabel('feature index')  # Label the feature axis.
plt.ylabel('coefficient')  # Label the coefficient axis.
plt.title('Lasso shrinks irrelevant coefficients toward zero')  # Title the plot.
plt.legend()  # Show coefficient labels.
plt.show()  # Render the coefficient plot.



## Exam-style problem prompt

A prompt gives many predictors and asks for prediction plus variable selection. Fit lasso with standardized predictors, choose alpha by CV, report test MSE, and identify nonzero coefficients.



    ## Adaptable code pattern

    ```python
    # Step 1: Split data into training and test sets.
# Step 2: Standardize predictors before applying the lasso penalty.
# Step 3: Use LassoCV or CV over alpha values.
# Step 4: Fit using training data only.
# Step 5: Predict test data and compute MSE.
# Step 6: Interpret alpha and nonzero coefficients.
    ```



## What to remember for the exam

Lasso is useful when variable selection is part of the question. Always mention standardization and the alpha tradeoff.
